In [1]:
import sys
print(sys.executable)

c:\Users\LEST\inclusion-financiere\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Toutes les bibliothèques sont installées !")

Toutes les bibliothèques sont installées !


In [4]:
from pathlib import Path

print("Dossier actuel :", Path.cwd())
print("\nContenu du dossier :")

for element in Path.cwd().iterdir():
    print(element.name)


Dossier actuel : c:\Users\LEST\inclusion-financiere\note book

Contenu du dossier :
01_exploration.ipynb
02_preparation_protocole.ipynb
03_baselines_ensembles.ipynb
04_hyperparametres.ipynb
05_calibration_seuil.ipynb
06_test_final_interpretation.ipynb


In [5]:
from pathlib import Path

projet = Path.cwd().parent

print("Dossier du projet :", projet)
print("\nContenu :")

for element in projet.iterdir():
    print(element.name)

Dossier du projet : c:\Users\LEST\inclusion-financiere

Contenu :
.git
.python-version
.venv
.vscode
Data
note book
notebook-test.ipynb
pyproject.toml
README.md
reports
src
tests
uv.lock


In [6]:
from pathlib import Path

projet = Path.cwd().parent
dossier_data = projet / "Data"

for fichier in dossier_data.rglob("*"):
    if fichier.is_file():
        print(fichier)

c:\Users\LEST\inclusion-financiere\Data\raw\inclusion-financiere-en-afrique.csv
c:\Users\LEST\inclusion-financiere\Data\raw\VariableDefinitions.csv


In [7]:
import pandas as pd
from pathlib import Path

chemin = Path.cwd().parent / "Data" / "raw" / "inclusion-financiere-en-afrique.csv"

print("Fichier trouvé :", chemin.exists())

df = pd.read_csv(chemin)

print("Dimensions :", df.shape)
df.head()

Fichier trouvé : True
Dimensions : (23524, 13)


,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3,24,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5,70,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5,26,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5,34,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8,26,Male,Child,Single/Never Married,Primary education,Informally employed


In [8]:
import pandas as pd
from inclusion_financiere.config import RAW
df = pd.read_csv(RAW)
df.info()
df["bank_account"].value_counts(normalize=True)

<class 'pandas.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   country                 23524 non-null  str  
 1   year                    23524 non-null  int64
 2   uniqueid                23524 non-null  str  
 3   bank_account            23524 non-null  str  
 4   location_type           23524 non-null  str  
 5   cellphone_access        23524 non-null  str  
 6   household_size          23524 non-null  int64
 7   age_of_respondent       23524 non-null  int64
 8   gender_of_respondent    23524 non-null  str  
 9   relationship_with_head  23524 non-null  str  
 10  marital_status          23524 non-null  str  
 11  education_level         23524 non-null  str  
 12  job_type                23524 non-null  str  
dtypes: int64(3), str(10)
memory usage: 2.3 MB


bank_account
No     0.859208
Yes    0.140792
Name: proportion, dtype: float64

In [13]:
%pip install -e ..

Note: you may need to restart the kernel to use updated packages.


c:\Users\LEST\inclusion-financiere\.venv\Scripts\python.exe: No module named pip


In [2]:
import sys
from pathlib import Path

# Retrouver la racine du projet
racine = Path.cwd().parent

# Ajouter le dossier src au chemin de recherche de Python
src = racine / "src"
sys.path.insert(0, str(src))

print("Racine du projet :", racine)
print("Dossier src trouvé :", src.exists())
print("Package trouvé :", (src / "inclusion_financiere").exists())

Racine du projet : c:\Users\LEST\inclusion-financiere
Dossier src trouvé : True
Package trouvé : True


In [3]:
from inclusion_financiere.data import load_raw, clean

print("Importation réussie !")

ModuleNotFoundError: No module named 'inclusion_financiere.data'

In [1]:
# src/inclusion_financiere/data.py
import pandas as pd
from .config import RAW, TARGET
CAT_COLS = ["country", "location_type", "cellphone_access", "gender_of_respondent",
"relationship_with_head", "marital_status", "education_level", "job_type"]

NUM_COLS = ["household_size", "age_of_respondent"]

def load_raw(path=RAW) -> pd.DataFrame:
return pd.read_csv(path)

def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
"""Sépare X et y ; retire les colonnes qui ne doivent pas servir de variables."""
y = (df[TARGET] == "Yes").astype(int)
X = df.drop(columns=[TARGET, "uniqueid", "year"])
return X, y

IndentationError: expected an indented block after function definition on line 9 (3250120837.py, line 10)

In [ ]:
# src/inclusion_financiere/features.py
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
EDU_ORDER = {"No formal education": 0, "Primary education": 1, "Secondary education": 2,

"Vocational/Specialised training": 3, "Tertiary education": 4}
FORMAL_JOBS = {"Formally employed Private", "Formally employed Government"}

class AddFeatures(BaseEstimator, TransformerMixin):
"""Feature engineering sans apprentissage : aucune statistique tirée des données."""
def fit(self, X, y=None):
return self
def transform(self, X):
X = X.copy()
X["education_rank"] = X["education_level"].map(EDU_ORDER).fillna(-1)
X["is_head"] = (X["relationship_with_head"] == "Head of Household").astype(int)
X["formal_job"] = X["job_type"].isin(FORMAL_JOBS).astype(int)
X["urban_phone"] = ((X["location_type"] == "Urban")

& (X["cellphone_access"] == "Yes")).astype(int)
X["age_group"] = pd.cut(X["age_of_respondent"], [15, 25, 35, 50, 65, 120],
labels=["16-25", "26-35", "36-50", "51-65",

"66+"]).astype(str)
return X

def onehot_preprocessor(cat_cols, num_cols):
return ColumnTransformer([
("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20),
cat_cols),
("num", StandardScaler(), num_cols),
])